In [ ]:
import pandas as pd
ips = pd.read_csv("C:/Users/Jyun/Downloads/testdatset_260427_ips.csv")
ips

In [ ]:
ips.payload.value_counts()

In [ ]:
for i in range(0, 2500):
    print(ips["full_log"].iloc[i])

In [ ]:
for i in range(2500, 5000):
    print(ips["full_log"].iloc[i])

In [ ]:
for i in range(5000, 7500):
    print(ips["full_log"].iloc[i])

In [ ]:
for i in range(7500, 10000):
    print(ips["full_log"].iloc[i])

In [ ]:
# 폴더에서 여러 csv 읽기
import glob
import os

file_path = glob.glob("C:/Users/Jyun/Downloads/ips_exd_result2/*.csv")
df_list = []

for file in file_path:
    try:
        # 1. 파일 읽기 옵션 강화
        # - skip_blank_lines=True: 내용이 없는 빈 줄은 알아서 무시하고 읽지 않음
        # - on_bad_lines='skip': 데이터가 쪼개져서 컬럼 개수가 안 맞는 에러 행은 버림
        # - engine='python': 복잡한 따옴표 안의 줄바꿈을 더 똑똑하게 처리함
        temp_df = pd.read_csv(
            file,
            skip_blank_lines=True,
            on_bad_lines="skip",
            engine="python",  # C 엔진보다 느리지만 텍스트 파싱에 더 견고합니다.
        )

        # 2. 모든 컬럼이 NaN(결측치)인 완벽한 빈 행 한 번 더 강제 삭제
        temp_df = temp_df.dropna(how="all")

        # 파일 이름 출처 기록 (선택 사항)
        temp_df["file_name"] = os.path.basename(file)

        df_list.append(temp_df)

    except Exception as e:
        print(f"[{os.path.basename(file)}] 파싱 실패: {e}")

# 3. 병합
ips_exd_result_raw = pd.concat(df_list, ignore_index=True)

print(f"합쳐진 파일 개수: {len(df_list)}개")
print(f"순수 데이터 건수: {len(ips_exd_result_raw)}건")

합쳐진 파일 개수: 10020개
순수 데이터 건수: 20000건


In [ ]:
ips_exd_result_raw

In [ ]:
ips_exd_result = ips_exd_result_raw.copy()

In [ ]:
import json


# JSON 파싱 함수 정의: 비고 컬럼에서 pcap, ai_prediction, ai_score를 추출
def extract_ai_features(json_data):
    # 1. 데이터가 비어있거나 결측치(NaN)인 경우 방어
    if pd.isna(json_data) or not json_data:
        return pd.Series([None, None, None])

    parsed_data = {}

    # 2. 데이터가 문자열(str)인 경우 파싱 시도
    if isinstance(json_data, str):
        try:
            # [핵심] strict=False 옵션 추가: payload 내부의 \n, \r 등 제어 문자로 인한 에러 무시
            parsed_data = json.loads(json_data, strict=False)
        except Exception as e:
            # 에러 원인 파악을 위해 에러 메시지와 데이터 일부를 출력
            print(f"JSON 파싱 에러: {e}")
            print(f"문제의 데이터 샘플: {json_data[:100]}...\n")
            return pd.Series([None, None, None])

    # 3. 데이터가 이미 딕셔너리(dict) 객체인 경우 (파싱 불필요)
    elif isinstance(json_data, dict):
        parsed_data = json_data

    else:
        return pd.Series([None, None, None])

    # 4. 값 추출 후 반환
    return pd.Series(
        [
            parsed_data.get("payload"),
            parsed_data.get("pcap"),
            parsed_data.get("ai_prediction"),
            parsed_data.get("ai_score"),
        ]
    )


# 추출한 데이터들을 새로운 컬럼으로 할당
ips_exd_result[["payload_exd", "pcap", "ai_prediction", "ai_score"]] = ips_exd_result[
    "비고"
].apply(extract_ai_features)

ips_exd_result

In [ ]:
ips_exd_result[ips_exd_result.payload_exd.isna()]

,로그소스 IP,로그소스명,발생시간,출발지 IP,출발지 포트,목적지 IP,목적지 포트,메소드,AI 예측,AI 스코어,...,경보 이름,경보 그룹,종료시간,지속시간,Log Type,AI 모델,payload_exd,pcap,ai_prediction,ai_score


In [ ]:
ips_exd_result.payload_exd.value_counts()

In [ ]:
ips_exd_result2 = ips_exd_result.drop_duplicates(subset=["payload_exd"])
ips_exd_result2.payload_exd.value_counts()

In [ ]:
ips_exd_result2.pcap.value_counts()

In [ ]:
# 대상 데이터셋(df_target)과 pcap을 기준으로 병합 (Left Join)
# how='left'는 df_target의 데이터를 모두 유지하면서 매칭되는 값만 붙이겠다는 의미입니다.
# 원하는 컬럼만 골라서 병합합니다.
df_merged = pd.merge(
    ips,
    ips_exd_result2[["pcap", "payload_exd", "ai_prediction", "ai_score"]],
    on="pcap",
    how="left",
)

print("--- 병합된 결과 ---")
print(df_merged[["pcap", "ai_prediction", "ai_score"]])

In [ ]:
df_merged

In [ ]:
df_merged[df_merged.ai_prediction.isna()]

,mgr_ip,event_time,origin,origin_name,s_info,s_loca,s_port,d_info,d_loca,d_port,...,total_packet,traffic_bps,traffic_pps,worktime,xstatus,xtime,full_log,payload_exd,ai_prediction,ai_score


In [ ]:
df_merged.columns

Index(['mgr_ip', 'event_time', 'origin', 'origin_name', 's_info', 's_loca',
       's_port', 'd_info', 'd_loca', 'd_port', 'protocol', 'status', 'logid',
       'method', 'hexa', 'xevent', 'attack_type', 'xdirection', 'RAW',
       'attack', 'category', 'cdtime', 'count', 'cpu_usages', 'd_addr',
       'd_addr6', 'd_country', 'd_dir', 'direction', 'endtime', 'etc_genio',
       'ext1', 'ext2', 'g_id', 'hacking_byte', 'haking_packet', 'hdd_use',
       'logtype', 'method1', 'mgr_time', 'network_drop', 'network_error',
       'note', 'origin_id', 'packet_icmp', 'packet_nonip', 'packet_tcp',
       'packet_udp', 'payload', 'pcap', 'pcap_dec', 'pkt_size', 'product',
       'risk', 's_addr', 's_addr6', 's_country', 's_dir', 'sniper_id',
       'total_kbyte', 'total_packet', 'traffic_bps', 'traffic_pps', 'worktime',
       'xstatus', 'xtime', 'full_log', 'payload_exd', 'ai_prediction',
       'ai_score'],
      dtype='object')

In [ ]:
df_merged.drop(
    columns=[
        "mgr_ip",
        "event_time",
        "origin",
        "origin_name",
        "s_loca",
        "s_port",
        "d_loca",
        "d_port",
        "logid",
        "hexa",
        "xevent",
        "attack_type",
        "xdirection",
        "RAW",
        "attack",
        "category",
        "cdtime",
        "count",
        "cpu_usages",
        "d_addr",
        "d_addr6",
        "d_country",
        "d_dir",
        "direction",
        "endtime",
        "etc_genio",
        "ext1",
        "ext2",
        "g_id",
        "hacking_byte",
        "haking_packet",
        "hdd_use",
        "logtype",
        "method1",
        "mgr_time",
        "network_drop",
        "network_error",
        "note",
        "origin_id",
        "packet_icmp",
        "packet_nonip",
        "packet_tcp",
        "packet_udp",
        "pcap_dec",
        "pkt_size",
        "product",
        "risk",
        "s_addr",
        "s_addr6",
        "s_country",
        "s_dir",
        "sniper_id",
        "total_kbyte",
        "total_packet",
        "traffic_bps",
        "traffic_pps",
        "worktime",
        "xstatus",
        "xtime",
    ],
    inplace=True,
)
df_merged.columns

Index(['s_info', 'd_info', 'protocol', 'status', 'method', 'payload', 'pcap',
       'full_log', 'payload_exd', 'ai_prediction', 'ai_score'],
      dtype='object')

In [ ]:
df_merged.rename(
    columns={"ai_prediction": "exd_ai_prediction", "ai_score": "exd_ai_score"},
    inplace=True,
)
df_merged

In [ ]:
df_merged.to_csv(
    "C:/Users/Jyun/Downloads/ips_exd_result_260429_le.csv", encoding="utf-8-sig"
)

In [ ]:
import pandas as pd
waf = pd.read_csv("C:/Users/Jyun/Downloads/testdataset_260427_waf.csv")
waf

In [ ]:
waf.full_log

In [ ]:
waf.payload.value_counts()

In [ ]:
# repr 함수: 데이터가 메모리에 담기 형태 그대로 출력(\n, \r 등을 그대로 출력)
for i in range(0, 2500):
    print(repr(waf["full_log"].loc[i]))

In [ ]:
for i in range(2500, 5000):
    print(repr(waf["full_log"].loc[i]))

In [ ]:
for i in range(5000, 7500):
    print(repr(waf["full_log"].loc[i]))

In [ ]:
for i in range(7500, 10000):
    print(repr(waf["full_log"].loc[i]))

In [ ]:
# 폴더에서 여러 csv 읽기
import glob
import os

file_path = glob.glob("C:/Users/Jyun/Downloads/waf_exd_result/*.csv")
df_list = []

for file in file_path:
    try:
        # 1. 파일 읽기 옵션 강화
        # - skip_blank_lines=True: 내용이 없는 빈 줄은 알아서 무시하고 읽지 않음
        # - on_bad_lines='skip': 데이터가 쪼개져서 컬럼 개수가 안 맞는 에러 행은 버림
        # - engine='python': 복잡한 따옴표 안의 줄바꿈을 더 똑똑하게 처리함
        temp_df = pd.read_csv(
            file,
            skip_blank_lines=True,
            on_bad_lines="skip",
            engine="python",  # C 엔진보다 느리지만 텍스트 파싱에 더 견고합니다.
        )

        # 2. 모든 컬럼이 NaN(결측치)인 완벽한 빈 행 한 번 더 강제 삭제
        temp_df = temp_df.dropna(how="all")

        # 파일 이름 출처 기록 (선택 사항)
        temp_df["file_name"] = os.path.basename(file)

        df_list.append(temp_df)

    except Exception as e:
        print(f"[{os.path.basename(file)}] 파싱 실패: {e}")

# 3. 병합
waf_exd_result_raw = pd.concat(df_list, ignore_index=True)

print(f"합쳐진 파일 개수: {len(df_list)}개")
print(f"순수 데이터 건수: {len(waf_exd_result_raw)}건")

합쳐진 파일 개수: 10011개
순수 데이터 건수: 15998건


In [ ]:
waf_exd_result_raw

In [ ]:
waf_exd_result = waf_exd_result_raw.copy()

In [ ]:
import json


# JSON 파싱 함수 정의: 비고 컬럼에서 pcap, ai_prediction, ai_score를 추출
def extract_ai_features(json_data):
    # 1. 데이터가 비어있거나 결측치(NaN)인 경우 방어
    if pd.isna(json_data) or not json_data:
        return pd.Series([None, None, None])

    parsed_data = {}

    # 2. 데이터가 문자열(str)인 경우 파싱 시도
    if isinstance(json_data, str):
        try:
            # [핵심] strict=False 옵션 추가: payload 내부의 \n, \r 등 제어 문자로 인한 에러 무시
            parsed_data = json.loads(json_data, strict=False)
        except Exception as e:
            # 에러 원인 파악을 위해 에러 메시지와 데이터 일부를 출력
            print(f"JSON 파싱 에러: {e}")
            print(f"문제의 데이터 샘플: {json_data[:100]}...\n")
            return pd.Series([None, None, None])

    # 3. 데이터가 이미 딕셔너리(dict) 객체인 경우 (파싱 불필요)
    elif isinstance(json_data, dict):
        parsed_data = json_data

    else:
        return pd.Series([None, None, None])

    # 4. 값 추출 후 반환
    return pd.Series(
        [
            parsed_data.get("payload"),
            parsed_data.get("ai_prediction"),
            parsed_data.get("ai_score"),
        ]
    )


# 추출한 데이터들을 새로운 컬럼으로 할당
waf_exd_result[["payload", "ai_prediction", "ai_score"]] = waf_exd_result[
    "비고"
].apply(extract_ai_features)

waf_exd_result

In [ ]:
waf_exd_result[waf_exd_result.payload.isna()]

,로그소스 IP,로그소스명,발생시간,출발지 IP,출발지 포트,목적지 IP,목적지 포트,메소드,AI 예측,AI 스코어,...,경보 유형,경보 이름,경보 그룹,종료시간,지속시간,Log Type,AI 모델,payload,ai_prediction,ai_score


In [ ]:
waf_exd_result.payload.value_counts()

In [ ]:
waf_exd_result2 = waf_exd_result.drop_duplicates(subset=["payload"])
waf_exd_result2.payload.value_counts()

In [ ]:
import codecs
# 텍스트로 굳어있는 이스케이프 문자(\n, \t, \x0b 등)를 실제 제어 기호로 변환하는 함수
def unescape_text(text):
    if pd.isna(text): 
        return ""
    try:
        return codecs.decode(str(text), 'unicode_escape')
    except Exception:
        return str(text)

# waf_exd_result2의 payload에 굳어있는 텍스트를 실제 기호로 먼저 풀어주기 (임시 컬럼 생성)
waf_exd_result2['temp_payload'] = waf_exd_result2['payload'].apply(unescape_text)

# 3. 양쪽 데이터에서 영문자와 숫자만 추출하여 완벽히 동일한 'join_key' 생성
# 이제 \x0b는 문자 x0b가 아니라 진짜 제어 기호가 되었으므로 정규식에 의해 완벽하게 삭제됩니다.
waf['join_key'] = waf['payload'].str.replace(r'[^a-zA-Z0-9]', '', regex=True)
waf_exd_result2['join_key'] = waf_exd_result2['temp_payload'].str.replace(r'[^a-zA-Z0-9]', '', regex=True)

In [ ]:
waf_exd_result2

In [ ]:
waf.join_key.value_counts()

In [ ]:
waf_exd_result2.join_key.value_counts()

In [ ]:
# 대상 데이터셋(df_target)과 payload를 기준으로 병합 (Left Join)
# how='left'는 df_target의 데이터를 모두 유지하면서 매칭되는 값만 붙이겠다는 의미입니다.
# 원하는 컬럼만 골라서 병합합니다.
df_merged = pd.merge(
    waf,
    waf_exd_result2[["join_key", "ai_prediction", "ai_score"]],
    on="join_key",
    how="left",
)

print("--- 병합된 결과 ---")
print(df_merged[["ai_prediction", "ai_score"]])

--- 병합된 결과 ---
      ai_prediction  ai_score
0            normal   76.2914
1            normal   99.9288
2            normal   76.2914
3            normal   99.9288
4            normal   99.9288
...             ...       ...
10003        normal   76.2914
10004        normal   59.3672
10005     anomalies   93.6564
10006     anomalies   99.8003
10007        normal  100.0000

[10008 rows x 2 columns]


In [ ]:
df_merged

In [ ]:
df_merged.payload.value_counts()

In [ ]:
df_merged.drop_duplicates(subset=['payload'], inplace=True)
df_merged

In [ ]:
df_merged[df_merged.ai_prediction.isna()]

In [ ]:
df_merged.columns

Index(['mgr_ip', 'event_time', 'origin', 'origin_name', 's_info', 's_loca',
       's_port', 'd_info', 'd_loca', 'd_port', 'protocol', 'status', 'logid',
       'method', 'hexa', 'xevent', 'attack_type', 'xdirection', 'RAW',
       'attack', 'category', 'cdtime', 'count', 'cpu_usages', 'd_addr',
       'd_addr6', 'd_country', 'd_dir', 'direction', 'endtime', 'etc_genio',
       'ext1', 'ext2', 'g_id', 'hacking_byte', 'haking_packet', 'hdd_use',
       'logtype', 'method1', 'mgr_time', 'network_drop', 'network_error',
       'note', 'origin_id', 'packet_icmp', 'packet_nonip', 'packet_tcp',
       'packet_udp', 'payload', 'pcap', 'pcap_dec', 'pkt_size', 'product',
       'risk', 's_addr', 's_addr6', 's_country', 's_dir', 'sniper_id',
       'total_kbyte', 'total_packet', 'traffic_bps', 'traffic_pps', 'worktime',
       'xstatus', 'xtime', 'full_log', 'join_key', 'ai_prediction',
       'ai_score'],
      dtype='object')

In [ ]:
df_merged.drop(
    columns=[
        "mgr_ip",
        "event_time",
        "origin",
        "origin_name",
        "s_loca",
        "s_port",
        "d_loca",
        "d_port",
        "logid",
        "hexa",
        "xevent",
        "attack_type",
        "xdirection",
        "RAW",
        "attack",
        "category",
        "cdtime",
        "count",
        "cpu_usages",
        "d_addr",
        "d_addr6",
        "d_country",
        "d_dir",
        "direction",
        "endtime",
        "etc_genio",
        "ext1",
        "ext2",
        "g_id",
        "hacking_byte",
        "haking_packet",
        "hdd_use",
        "logtype",
        "method1",
        "mgr_time",
        "network_drop",
        "network_error",
        "note",
        "origin_id",
        "packet_icmp",
        "packet_nonip",
        "packet_tcp",
        "packet_udp",
        "pcap_dec",
        "pkt_size",
        "product",
        "risk",
        "s_addr",
        "s_addr6",
        "s_country",
        "s_dir",
        "sniper_id",
        "total_kbyte",
        "total_packet",
        "traffic_bps",
        "traffic_pps",
        "worktime",
        "xstatus",
        "xtime",
    ],
    inplace=True,
)
df_merged.columns

Index(['s_info', 'd_info', 'protocol', 'status', 'method', 'payload', 'pcap',
       'full_log', 'join_key', 'ai_prediction', 'ai_score'],
      dtype='object')

In [ ]:
df_merged.rename(
    columns={"ai_prediction": "exd_ai_prediction", "ai_score": "exd_ai_score"},
    inplace=True,
)
df_merged

In [ ]:
df_merged.to_csv(
    "C:/Users/Jyun/Downloads/waf_exd_result_260429.csv", encoding="utf-8-sig"
)